# 02 · Población base 2025

Arma la población de partida de la microsimulación a partir de la EPH continua del
3.er trimestre de 2025, agregada en celdas de
`children × sex × status × provincia × education × age × clima_educativo` con su peso de
expansión `n`.

**Decisiones de construcción**

- **Universo urbano, sin calibración externa:** las edades 1 a 100 quedan tal como las expande
  la EPH. Se descartan las edades mayores a 100 (la grilla de mortalidad termina ahí).
- **Edad 0 descartada:** en el corte 2025 la cohorte de edad 0 tiene un déficit concentrado en
  el GBA (problema de composición, no de nivel). Se regenera endógenamente en el notebook 04
  con el mismo motor que produce las cohortes 2026-2075.
- **`status` = tercil de ingreso por adulto** (`itf` / miembros de 18 años o más), ponderado por
  `pondih` sobre los hogares con respuesta de ingresos. La no respuesta se imputa con un
  *hot-deck* ponderado jerárquico (educación × sexo × clima × provincia → educación × sexo →
  global), con semilla fija.
- **Clima educativo del hogar:** promedio de años de educación de los miembros de 18+ en cinco
  categorías (cortes INDEC: <7, 7-11, 11-14, 14-16, 16+).
- **Paridad (`children`):** hijos convivientes del jefe/cónyuge o nietos adjudicados a la hija
  mayor biológicamente plausible, menos el nacimiento del año; es la misma definición con la
  que se entrena el modelo de fecundidad.

**Entrada:** `datos/eph/personas_tot.urb_3T_2025.txt`.

**Salida:** `resultados/intermedios/poblacion_base/poblacion_base_sin_edad0.csv`
(el notebook 04 le agrega la cohorte de edad 0 y escribe `poblacion_base.csv`).

**Tiempo aproximado:** 10 segundos.

In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
DIR_POB = Path("resultados/intermedios/poblacion_base")
DIR_POB.mkdir(parents=True, exist_ok=True)

N_ESTRATOS_STATUS = 3
MIN_DONANTES_IMPUTACION = 10
SEMILLA_IMPUTACION = 20260905
EDAD_MIN_MATERNIDAD = 15
EDAD_MAX_BASE = 100

dict_provincias = {32: 1, 33: 2, 2: 2, 3: 2, 34: 2, 38: 2, 40: 2, 18: 3, 61: 3, 19: 4, 48: 4,
                   22: 5, 41: 5, 23: 6, 55: 6, 25: 7, 50: 7, 29: 8, 62: 8, 7: 9, 52: 9,
                   8: 10, 44: 10, 12: 11, 43: 11, 15: 12, 47: 12, 10: 13, 51: 13, 26: 14, 57: 14,
                   27: 15, 56: 15, 4: 16, 5: 16, 60: 16, 6: 17, 14: 17, 46: 17, 13: 18, 36: 18,
                   42: 18, 30: 19, 49: 19, 9: 20, 91: 20, 45: 20, 17: 21, 53: 21, 20: 22, 58: 22,
                   31: 23, 93: 24, 54: 24}
provincia_region = {1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3, 12: 3,
                    13: 4, 14: 4, 15: 4, 16: 5, 17: 5, 18: 5, 19: 5, 20: 6, 21: 6, 22: 6, 23: 6,
                    24: 6}

In [2]:
def get_generation(birth_year):
    conditions = [birth_year.between(1946, 1964), birth_year.between(1965, 1980),
                  birth_year.between(1981, 1996), birth_year.between(1997, 2012),
                  birth_year.between(2013, 2025)]
    return np.select(conditions, [1, 2, 3, 4, 5], default=0)


def get_ingreso_adulto(df, col_hogar='hogar_id', col_edad='ch06'):
    itf = pd.to_numeric(df['itf'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    adultos = df.groupby(col_hogar)[col_edad].transform(lambda s: (s >= 18).sum()).clip(lower=1)
    return itf / adultos


def get_weighted_quantiles(df, value_col, weight_col, k=N_ESTRATOS_STATUS):
    valid = df[value_col].notna() & df[weight_col].notna() & (df[weight_col] > 0)
    df_valid = df[valid].sort_values(value_col, kind='mergesort')
    cum = df_valid[weight_col].cumsum()
    q = pd.cut(cum / cum.iloc[-1], bins=np.linspace(0, 1, k + 1),
               labels=range(1, k + 1), include_lowest=True)
    return q.astype('Int64').reindex(df.index)


def imputar_status(df, seed=SEMILLA_IMPUTACION, k=N_ESTRATOS_STATUS, min_donantes=MIN_DONANTES_IMPUTACION):
    rng = np.random.default_rng(seed)
    st = pd.to_numeric(df['status'], errors='coerce').to_numpy(dtype=float)
    w = pd.to_numeric(df['pondih'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
    for keys in [['education', 'sex', 'clima_educativo', 'provincia'], ['education', 'sex'], None]:
        if not np.isnan(st).any():
            break
        grupos = ({None: np.arange(len(df))} if keys is None
                  else df.groupby(keys, dropna=False, observed=True).indices)
        for pos in grupos.values():
            pos = np.asarray(pos)
            falta = np.isnan(st[pos])
            if not falta.any():
                continue
            don = pos[~falta]
            if len(don) < min_donantes:
                continue
            pw = w[don]
            if not np.isfinite(pw).all() or pw.sum() <= 0:
                pw = np.ones(len(don))
            st[pos[falta]] = rng.choice(st[don], size=int(falta.sum()), replace=True, p=pw / pw.sum())
    resto = np.isnan(st)
    if resto.any():
        st[resto] = rng.integers(1, k + 1, size=int(resto.sum()))
    return pd.Series(st, index=df.index).round().astype(int)


def asignar_nietos_a_hijas(df, edad_col):
    idx = df.index
    vacio_n, vacio_age, vacio_m = pd.Series(0.0, index=idx), pd.Series(99.0, index=idx), pd.Series(0, index=idx)
    nietos = (df.loc[df['ch03'] == 5, ['hogar_id', edad_col]].rename(columns={edad_col: 'edad_nieto'})
              .reset_index().rename(columns={'index': 'idx_nieto'}))
    hijas = (df.loc[(df['ch03'] == 3) & (df['sex'] == 1) & (df[edad_col].between(15, 50)), ['hogar_id', edad_col]]
             .rename(columns={edad_col: 'edad_hija'}).reset_index().rename(columns={'index': 'idx_hija'}))
    if nietos.empty or hijas.empty:
        return vacio_n, vacio_age, vacio_m
    pares = nietos.merge(hijas, on='hogar_id', how='inner')
    pares = pares[pares['edad_hija'] >= pares['edad_nieto'] + EDAD_MIN_MATERNIDAD]
    if pares.empty:
        return vacio_n, vacio_age, vacio_m
    pares = pares.sort_values(['idx_nieto', 'edad_hija', 'idx_hija'], ascending=[True, False, True])
    adjudicado = pares.drop_duplicates(subset='idx_nieto', keep='first')
    n_asig = adjudicado.groupby('idx_hija').size()
    min_age = adjudicado.groupby('idx_hija')['edad_nieto'].min()
    nietos_asignados = pd.Series(idx.map(n_asig), index=idx).astype(float).fillna(0.0)
    min_nieto_asig = pd.Series(idx.map(min_age), index=idx).astype(float).fillna(99.0)
    return nietos_asignados, min_nieto_asig, (min_nieto_asig == 0).astype(int)

In [3]:
df = pd.read_csv('datos/eph/personas_tot.urb_3T_2025.txt', sep=';', low_memory=False)
df.columns = df.columns.str.lower()

df['ch06'] = df['ch06'].astype(int).replace(-1, 0)
df['nivel_ed'] = df['nivel_ed'].astype(int).replace(7, 0)
df = df.rename(columns={'ch06': 'age'})
df['hogar_id'] = df['codusu'] + '_' + df['nro_hogar'].astype(str) + '_' + df['ano4'].astype(str)
df['education'] = df['nivel_ed'].astype(int)
df['sex'] = df['ch04'].map({2: 1, 1: 0})
df['provincia'] = df['aglomerado'].map(dict_provincias).fillna(0).astype(int)
for c in ['ch10', 'ch12', 'ch13', 'ch14']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# años de educación según nivel cursado (ch12), si lo terminó (ch13) y último año aprobado (ch14)
nivel = df['ch12'].fillna(0).astype(int)
finalizado = df['ch13'] == 1
aprob = pd.to_numeric(df['ch14'], errors='coerce').replace([98, 99], np.nan)
educ = df['education']
primaria, egb, secundaria, terciario, universidad, posgrado = 6, 9, 12, 14, 16, 18
sin_nivel = nivel.isin([9, 99]) | nivel.isna()
condiciones = [(nivel == 1),
               (nivel == 2) & finalizado, (nivel == 2) & aprob.notna(), (nivel == 2) & (educ == 2), (nivel == 2),
               (nivel == 3) & finalizado, (nivel == 3) & aprob.notna(), (nivel == 3) & (educ == 2), (nivel == 3),
               (nivel == 4) & finalizado, (nivel == 4) & aprob.notna(), (nivel == 4) & (educ == 4), (nivel == 4),
               (nivel == 5) & finalizado, (nivel == 5) & aprob.notna(), (nivel == 5) & (educ == 4), (nivel == 5),
               (nivel == 6) & finalizado, (nivel == 6) & aprob.notna(), (nivel == 6) & (educ == 6), (nivel == 6),
               (nivel == 7) & finalizado, (nivel == 7) & aprob.notna(), (nivel == 7) & (educ == 6), (nivel == 7),
               (nivel == 8) & finalizado, (nivel == 8) & aprob.notna(), (nivel == 8),
               sin_nivel & (educ == 1), sin_nivel & (educ == 2), sin_nivel & (educ == 3),
               sin_nivel & (educ == 4), sin_nivel & (educ == 5), sin_nivel & (educ == 6)]
resultados = [0, primaria, np.minimum(aprob, primaria - 0.5), primaria, primaria / 2,
              egb, np.minimum(aprob, egb - 0.5), primaria, egb / 2,
              secundaria, np.minimum(primaria + aprob, secundaria - 0.5), secundaria, primaria + 3,
              secundaria, np.minimum(egb + aprob, secundaria - 0.5), secundaria, egb + 1.5,
              terciario, np.minimum(secundaria + aprob, terciario - 0.5), terciario, secundaria + 1,
              universidad, np.minimum(secundaria + aprob, universidad - 0.5), universidad, secundaria + 2.5,
              posgrado, np.minimum(universidad + aprob, posgrado - 0.5), universidad + 1,
              primaria / 2, primaria, primaria + 3, secundaria, secundaria + 2, universidad]
df['in_system'] = (df['ch10'] == 1).astype(int)
df['ed_years'] = np.select(condiciones, resultados, default=0)

clima_hogar = df[df['age'] >= 18].groupby('hogar_id')['ed_years'].mean().reset_index(name='average_years')
df = df.merge(clima_hogar, on='hogar_id', how='left')
df['clima_educativo'] = (pd.cut(df['average_years'], bins=[-np.inf, 7, 11, 14, 16, np.inf],
                                labels=[0, 1, 2, 3, 4], right=False).fillna(0).astype(int))
df = df.merge(df.groupby('hogar_id').size().reset_index(name='hh_size'), on='hogar_id', how='left')

df['ipcf'] = pd.to_numeric(df['ipcf'], errors='coerce')
df['pondih'] = pd.to_numeric(df['pondih'], errors='coerce').fillna(0)
df['ingreso_adulto'] = get_ingreso_adulto(df, col_edad='age')
df['status'] = get_weighted_quantiles(df, 'ingreso_adulto', 'pondih')
df['status'] = imputar_status(df)

In [4]:
df = df.reset_index(drop=True)
hijos_jefe = (df[df['ch03'] == 3].groupby('hogar_id')
              .agg(children_jefe=('age', 'count'), min_child_age_jefe=('age', 'min')).reset_index())
df = df.merge(hijos_jefe, on='hogar_id', how='left')
df['children_jefe'] = df['children_jefe'].fillna(0)

nietos_asig, min_nieto_asig, _ = asignar_nietos_a_hijas(df, edad_col='age')
cond_jefa = df['ch03'].isin([1, 2]) & (df['sex'] == 1)
cond_hija = (df['ch03'] == 3) & (df['sex'] == 1) & (df['age'].between(15, 50))
target_children = np.where(cond_jefa, df['children_jefe'], np.where(cond_hija, nietos_asig.to_numpy(), 0.0))
youngest_age = np.where(cond_jefa, df['min_child_age_jefe'].fillna(99).to_numpy(),
                        np.where(cond_hija, min_nieto_asig.to_numpy(), 99.0))
target_motherhood = (youngest_age == 0).astype(int)
df['children'] = np.clip(target_children - target_motherhood, 0, None).astype(int)

In [5]:
columnas_agrupamiento = ['children', 'sex', 'status', 'provincia', 'education', 'age', 'clima_educativo']
poblacion_base = df.groupby(columnas_agrupamiento)['pondera'].sum().reset_index(name='n')
poblacion_base['n'] = poblacion_base['n'].fillna(0).round().astype(int)
poblacion_base = poblacion_base[poblacion_base['n'] > 0].copy()

edad0 = poblacion_base.loc[poblacion_base['age'] == 0, 'n'].sum()
poblacion_base = poblacion_base[poblacion_base['age'] > 0].copy()
print(f'[edad 0] se descartan {edad0:,} personas de la cohorte 2025 (se regenera en el notebook 04)')

poblacion_base['birth_year'] = 2025 - poblacion_base['age']
poblacion_base['generation'] = get_generation(poblacion_base['birth_year'])
poblacion_base['region'] = poblacion_base['provincia'].map(provincia_region)
for col in poblacion_base.columns:
    poblacion_base[col] = pd.to_numeric(poblacion_base[col], downcast='integer')

poblacion_base = poblacion_base[poblacion_base['age'] <= EDAD_MAX_BASE].copy()
assert poblacion_base['age'].between(1, EDAD_MAX_BASE).all()
assert (poblacion_base['n'] > 0).all()
assert poblacion_base['provincia'].between(1, 24).all()
assert poblacion_base['status'].between(1, N_ESTRATOS_STATUS).all()

poblacion_base.to_csv(DIR_POB / 'poblacion_base_sin_edad0.csv', index=False)
print(f'[export] poblacion_base_sin_edad0.csv -> {len(poblacion_base):,} celdas, '
      f'{poblacion_base["n"].sum():,} personas')

[edad 0] se descartan 283,030 personas de la cohorte 2025 (se regenera en el notebook 04)
[export] poblacion_base_sin_edad0.csv -> 46,958 celdas, 43,051,031 personas
